Optimization Goal: Maximise

Descriptions of sample application:

Detect likely contamination sources in a two-dimensional area, such as a radiation field, where only proximity yields a non-zero reading. The system uses Bayesian optimisation to tune detection parameters and reliably identify both strong and weak sources.

In [1]:
# =============================================================================
# IMPORTS
# =============================================================================

import warnings

import numpy as np
import pandas as pd
from scipy.stats import norm
from sklearn.exceptions import ConvergenceWarning
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, RBF

# Local GP fits regularly report optimiser convergence warnings (the length
# scales sit near their bounds); they do not change the fitted model.
warnings.filterwarnings("ignore", category=ConvergenceWarning)

In [2]:
# =============================================================================
# SETTINGS
# =============================================================================
# Function 1 objective values span more than 120 orders of magnitude, so a
# global model is not useful. The pipeline fits one small RBF GP on a
# signed-log transform of y, using only observations near the incumbent.
# Transform and kernel are fixed from the earlier model-selection rounds.

DATA_PATH = "../data/data.csv"
FUNCTION_ID = 1

# Signed-log target transform: sign(y) * log1p(|y| / scale)
LOCAL_SCALE_QUANTILE = 0.452591   # quantile of |y| that sets the scale (frozen from the R11 selection)

# Local region around the incumbent (unit-cube half-widths)
LOCAL_TRAIN_RADIUS = 0.04         # observations inside this normalised radius train the GP
LOCAL_SEARCH_RADIUS = 0.02        # acquisition candidates are searched inside this box

# Local GP (constant * ARD RBF)
LOCAL_GP_ALPHA = 1e-8
LOCAL_GP_RESTARTS = 20
LOCAL_GP_RANDOM_SEED = 42
LOCAL_GP_NORMALIZE_Y = True
LOCAL_LENGTH_SCALE_BOUNDS = (1e-3, 8e-2)
LOCAL_CONSTANT_BOUNDS = (1e-3, 1e3)

# Acquisition functions
ACQUISITION_GRID_RESOLUTION = 250   # grid points per axis in the search box
EI_XI = 0.01
UCB_KAPPA = 1.96

# Local validation
VALIDATION_GP_RESTARTS = 10
VALIDATION_GP_SEED = 42
STABILITY_TRAIN_RADII = [0.03, 0.04, 0.05, 0.06]
STABILITY_GRID_RESOLUTION = 150

# Trajectory: the validated rounds that moved steadily uphill
TRAJECTORY_ROUNDS = [6, 7, 9, 10, 11]   # add future rounds only when they follow the same trajectory
TRAJECTORY_GRID_POINTS = 1001
TRAJECTORY_T_MIN, TRAJECTORY_T_MAX = 0.0, 2.0
EXTENDED_T_MIN, EXTENDED_T_MAX, EXTENDED_GRID_POINTS = 0.0, 5.0, 5001

# 2-D exploration around the incumbent
EXPLORATION_RADII = [0.0005, 0.001, 0.002, 0.005]
RECOMMENDED_DIRECTION, RECOMMENDED_RADIUS = "Perpendicular -", 0.0005   # the row submitted this week

In [3]:
# =============================================================================
# DATA LOADING
# =============================================================================
# Loads the centrally prepared BBO dataset and selects the active function.
# Active input dimensions are inferred from the x-columns that contain data.

data = pd.read_csv(DATA_PATH, float_precision="round_trip")
function_data = data[data["function"] == FUNCTION_ID].copy()

feature_cols = [
    f"x{i}" for i in range(1, 9)
    if function_data[f"x{i}"].notna().any()
]

# Modelling arrays
X = function_data[feature_cols].to_numpy()
y = function_data["y"].to_numpy()
n_features = X.shape[1]

# Observation table
df = function_data[[*feature_cols, "y", "round"]].copy()
df["Stage"] = df["round"].apply(
    lambda r: "Initial Data" if r == 0 else f"Week {r}"
)
df = df.rename(columns={"round": "Round"})
df = df[[*feature_cols, "y", "Stage", "Round"]]

print(f"Function {FUNCTION_ID} observations: {len(df)}")
print(f"Active features: {feature_cols}")
print(f"Rounds: R0–R{df['Round'].max()}")
print(f"y range: {y.min():.3e} to {y.max():.3e} "
      f"({np.ptp(np.log10(np.abs(y[y != 0]))):.0f} orders of magnitude in |y|)")

Function 1 observations: 21
Active features: ['x1', 'x2']
Rounds: R0–R11
y range: -1.509e-02 to 3.546e-06 (122 orders of magnitude in |y|)


In [4]:
# =============================================================================
# LOCAL TARGET TRANSFORMATION
# =============================================================================
# Signed-log is monotone, so it preserves the maximisation ordering. The scale
# is re-estimated from the current data at the frozen quantile; refitting it as
# observations are added is intentional and does not leak a held-out label.

y_original = y
selected_scale = float(np.quantile(np.abs(y_original[y_original != 0]), LOCAL_SCALE_QUANTILE))


def transform_target(values):
    """Signed-log transform: sign(y) * log1p(|y| / scale)."""
    v = np.asarray(values, dtype=np.float64)
    return np.sign(v) * np.log1p(np.abs(v) / selected_scale)


def inverse_target(values):
    """Back-transform from the signed-log scale to y."""
    v = np.asarray(values, dtype=np.float64)
    return np.sign(v) * selected_scale * np.expm1(np.abs(v))


def fit_local_gp(X_fit, y_fit, restarts, seed):
    """Fit the constant * ARD RBF GP to signed-log targets."""
    kernel = ConstantKernel(1.0, constant_value_bounds=LOCAL_CONSTANT_BOUNDS) * RBF(
        length_scale=np.full(n_features, LOCAL_SEARCH_RADIUS),
        length_scale_bounds=LOCAL_LENGTH_SCALE_BOUNDS,
    )
    gp = GaussianProcessRegressor(kernel=kernel, alpha=LOCAL_GP_ALPHA,
                                  normalize_y=LOCAL_GP_NORMALIZE_Y,
                                  n_restarts_optimizer=restarts, random_state=seed)
    return gp.fit(X_fit, transform_target(y_fit))


def format_point(x):
    """Format a point as the dash-separated six-decimal submission string."""
    return "-".join(f"{v:.6f}" for v in x)


print(f"Signed-log scale (|y| quantile {LOCAL_SCALE_QUANTILE}): {selected_scale:.6e}")

Signed-log scale (|y| quantile 0.452591): 7.960436e-19


In [5]:
# =============================================================================
# LOCAL TRUST REGION AND LOCAL GP
# =============================================================================
# The training neighbourhood and the search box are set by explicit radii
# around the current incumbent (not by any global GP length scales).

X_model = X
incumbent_idx = np.argmax(y_original)
incumbent_x = X_model[incumbent_idx].copy()
incumbent_y = y_original[incumbent_idx]

# Training neighbourhood: normalised Euclidean distance from the incumbent
train_distances = np.linalg.norm((X_model - incumbent_x) / LOCAL_TRAIN_RADIUS, axis=1)
local_train_mask = train_distances <= 1.0

# Search box
lower_bounds = np.maximum(incumbent_x - LOCAL_SEARCH_RADIUS, 0.0)
upper_bounds = np.minimum(incumbent_x + LOCAL_SEARCH_RADIUS, 1.0)

print(f"Incumbent: x = {np.round(incumbent_x, 6)} | y = {incumbent_y:.16e}")
print(f"Local training radius {LOCAL_TRAIN_RADIUS} -> {local_train_mask.sum()} observations")
print("Search box: " + ", ".join(f"{f} [{lo:.6f}, {hi:.6f}]" for f, lo, hi in zip(feature_cols, lower_bounds, upper_bounds)))

# Local GP on the training neighbourhood
X_local, y_local_original = X_model[local_train_mask], y_original[local_train_mask]
with warnings.catch_warnings(record=True) as local_fit_warnings:
    warnings.simplefilter("always")
    gp_local = fit_local_gp(X_local, y_local_original, LOCAL_GP_RESTARTS, LOCAL_GP_RANDOM_SEED)

local_length_scales = np.atleast_1d(gp_local.kernel_.k2.length_scale)
print(f"\nLocal GP fitted kernel: {gp_local.kernel_}")
print(f"Log-marginal likelihood: {gp_local.log_marginal_likelihood_value_:.6f} | fit warnings: {len(local_fit_warnings)}")
print("ARD length scales: " + ", ".join(f"{f}={ls:.6f}" for f, ls in zip(feature_cols, local_length_scales)))
print("\nObservations in the training neighbourhood (nearest first):")
local_obs = df.loc[local_train_mask, [*feature_cols, "y", "Stage", "Round"]].copy()
local_obs["Distance"] = train_distances[local_train_mask]
print(local_obs.sort_values("Distance").to_string(index=False))

Incumbent: x = [0.67155 0.70077] | y = 3.5455529902955166e-06
Local training radius 0.04 -> 8 observations
Search box: x1 [0.651550, 0.691550], x2 [0.680770, 0.720770]

Local GP fitted kernel: 1.51**2 * RBF(length_scale=[0.0774, 0.0116])
Log-marginal likelihood: 17.758757 | fit warnings: 2
ARD length scales: x1=0.077352, x2=0.011602

Observations in the training neighbourhood (nearest first):
      x1       x2             y        Stage  Round  Distance
0.671550 0.700770  3.545553e-06      Week 11     11  0.000000
0.671200 0.700600  3.502779e-06      Week 10     10  0.009728
0.670800 0.700400  3.410778e-06       Week 9      9  0.020908
0.670400 0.700200  3.259175e-06       Week 7      7  0.032088
0.670000 0.700000  3.037630e-06       Week 6      6  0.043268
0.693467 0.708543  7.172332e-08       Week 5      5  0.581364
0.650114 0.681526 -3.606063e-03 Initial Data      0  0.720164
0.703200 0.715500  1.388951e-09       Week 4      4  0.872745


In [6]:
# =============================================================================
# LOCAL ACQUISITION FUNCTIONS
# =============================================================================
# Posterior mean, EI, PI and UCB on a grid over the search box. The winners
# all land on the box boundary, where the GP extrapolates and predicts
# implausibly large improvements, so they are NOT used as candidates; the
# trajectory and exploration analyses below are used instead.

g1, g2 = np.meshgrid(np.linspace(lower_bounds[0], upper_bounds[0], ACQUISITION_GRID_RESOLUTION),
                     np.linspace(lower_bounds[1], upper_bounds[1], ACQUISITION_GRID_RESOLUTION))
X_candidates = np.column_stack([g1.ravel(), g2.ravel()])

mu, sigma = gp_local.predict(X_candidates, return_std=True)
incumbent_transformed = transform_target([incumbent_y])[0]

improvement = mu - incumbent_transformed - EI_XI
z = np.divide(improvement, sigma, out=np.zeros_like(improvement), where=sigma > 0)
ei = improvement * norm.cdf(z) + sigma * norm.pdf(z)
ei[sigma == 0] = 0.0
pi = norm.cdf(z)
pi[sigma == 0] = (improvement[sigma == 0] > 0).astype(float)
ucb = mu + UCB_KAPPA * sigma

records = []
for method, score in [("Posterior Mean", mu), ("EI", ei), ("PI", pi), ("UCB", ucb)]:
    i = int(np.argmax(score))
    x_c = X_candidates[i]
    edges = [f"{f}:{'LOW' if np.isclose(v, lo, atol=1e-8) else 'HIGH'}"
             for f, v, lo, hi in zip(feature_cols, x_c, lower_bounds, upper_bounds)
             if np.isclose(v, lo, atol=1e-8) or np.isclose(v, hi, atol=1e-8)]
    records.append({"Method": method, **dict(zip(feature_cols, x_c)),
                    "Pred_Transformed": mu[i], "Pred_y": inverse_target([mu[i]])[0], "Pred_Std": sigma[i],
                    "EI": ei[i], "PI": pi[i], "UCB": ucb[i],
                    "Distance_to_Incumbent": np.linalg.norm(x_c - incumbent_x),
                    "On_Boundary": ", ".join(edges)})
acquisition_summary = pd.DataFrame(records)

print(f"Search grid: {ACQUISITION_GRID_RESOLUTION} x {ACQUISITION_GRID_RESOLUTION} = {len(X_candidates):,} points | EI xi {EI_XI} | UCB kappa {UCB_KAPPA}")
print(acquisition_summary.to_string(index=False))

Search grid: 250 x 250 = 62,500 points | EI xi 0.01 | UCB kappa 1.96
        Method      x1       x2  Pred_Transformed   Pred_y  Pred_Std       EI       PI       UCB  Distance_to_Incumbent On_Boundary
Posterior Mean 0.69155 0.699726         36.059658 0.003643  6.227899 7.342233 0.866911 48.266341               0.020027     x1:HIGH
            EI 0.69155 0.699405         36.035559 0.003556  6.469552 7.374745 0.856935 48.715881               0.020047     x1:HIGH
            PI 0.69155 0.701814         34.983385 0.001242  4.662798 6.082332 0.895134 44.122468               0.020027     x1:HIGH
           UCB 0.69155 0.697156         34.189305 0.000561  8.169691 6.391095 0.731939 50.201900               0.020324     x1:HIGH


In [7]:
# =============================================================================
# LOCAL MODEL VALIDATION
# =============================================================================
# 1. Leave-one-out on the local training set: does the local GP predict the
#    held-out signed-log value, and its sign?
# 2. Stability: how the length scales and the PI optimum move as the training
#    radius changes.

# 1. Local leave-one-out
local_indices = np.where(local_train_mask)[0]
loo_records = []
for i in range(len(y_local_original)):
    keep = np.arange(len(y_local_original)) != i
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        gp_loo = fit_local_gp(X_local[keep], y_local_original[keep], VALIDATION_GP_RESTARTS, VALIDATION_GP_SEED)
    pred, pred_std = gp_loo.predict(X_local[i].reshape(1, -1), return_std=True)
    actual_t = transform_target([y_local_original[i]])[0]
    loo_records.append({"Round": int(df["Round"].iloc[local_indices[i]]),
                        **dict(zip(feature_cols, X_local[i])),
                        "Actual_y": y_local_original[i], "Actual_Transformed": actual_t,
                        "Predicted_Transformed": pred[0], "Predicted_Std": pred_std[0],
                        "Standardised_Residual": (actual_t - pred[0]) / pred_std[0] if pred_std[0] > 0 else np.nan,
                        "Sign_Correct": bool(np.sign(y_local_original[i]) == np.sign(pred[0]))})
print("LOCAL GP LEAVE-ONE-OUT")
print(pd.DataFrame(loo_records).to_string(index=False))

# 2. Stability to the training radius
stability_records = []
for radius in STABILITY_TRAIN_RADII:
    mask = np.linalg.norm((X_model - incumbent_x) / radius, axis=1) <= 1.0
    X_fit, y_fit = X_model[mask], y_original[mask]
    if len(y_fit) < 3 or len(np.unique(np.sign(y_fit))) < 2:
        stability_records.append({"Train_Radius": radius, "N_Local": len(y_fit)})
        continue
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        gp_test = fit_local_gp(X_fit, y_fit, VALIDATION_GP_RESTARTS, VALIDATION_GP_SEED)
    ls = np.atleast_1d(gp_test.kernel_.k2.length_scale)
    axis = np.linspace(-LOCAL_SEARCH_RADIUS, LOCAL_SEARCH_RADIUS, STABILITY_GRID_RESOLUTION)
    dx1, dx2 = np.meshgrid(axis, axis)
    grid = np.column_stack([incumbent_x[0] + dx1.ravel(), incumbent_x[1] + dx2.ravel()])
    grid = grid[np.all((grid >= lower_bounds) & (grid <= upper_bounds), axis=1)]
    mu_t, sigma_t = gp_test.predict(grid, return_std=True)
    z_t = np.divide(mu_t - transform_target(y_fit).max(), sigma_t, out=np.full_like(mu_t, -np.inf), where=sigma_t > 0)
    pi_t = norm.cdf(z_t)
    best = int(np.argmax(pi_t))
    stability_records.append({"Train_Radius": radius, "N_Local": len(y_fit),
                              "x1_LengthScale": ls[0], "x2_LengthScale": ls[1],
                              "PI_x1": grid[best, 0], "PI_x2": grid[best, 1], "PI_Max": pi_t[best],
                              "PI_Distance_From_Incumbent": np.linalg.norm(grid[best] - incumbent_x)})
print("\nLOCAL MODEL STABILITY TO TRAINING RADIUS")
print(pd.DataFrame(stability_records).to_string(index=False))

LOCAL GP LEAVE-ONE-OUT
 Round       x1       x2      Actual_y  Actual_Transformed  Predicted_Transformed  Predicted_Std  Standardised_Residual  Sign_Correct
     0 0.650114 0.681526 -3.606063e-03          -36.049494              27.371748       3.385520             -18.733085         False
     4 0.703200 0.715500  1.388951e-09           21.279916              26.770025       8.617224              -0.637109          True
     5 0.693467 0.708543  7.172332e-08           25.224183              22.742693       2.107435               1.177493          True
     6 0.670000 0.700000  3.037630e-06           28.970200              28.980368       0.006507              -1.562686          True
     7 0.670400 0.700200  3.259175e-06           29.040597              29.037756       0.001940               1.463930          True
     9 0.670800 0.700400  3.410778e-06           29.086063              29.086411       0.002236              -0.155592          True
    10 0.671200 0.700600  3.502779e-06 


LOCAL MODEL STABILITY TO TRAINING RADIUS
 Train_Radius  N_Local  x1_LengthScale  x2_LengthScale    PI_x1    PI_x2   PI_Max  PI_Distance_From_Incumbent
         0.03        7        0.080000        0.011536 0.673832 0.700904 0.764656                    0.002286
         0.04        8        0.077355        0.011602 0.691550 0.701710 0.895538                    0.020022
         0.05        8        0.077355        0.011602 0.691550 0.701710 0.895538                    0.020022
         0.06        8        0.077355        0.011602 0.691550 0.701710 0.895538                    0.020022


In [8]:
# =============================================================================
# LOCAL TRAJECTORY ANALYSIS
# =============================================================================
# The high-value observations lie almost on one line (R6 -> R7 -> R9 -> R10 ->
# R11). The trajectory direction is fitted to those observations (first
# principal component), then:
#   - a quadratic in y along the trajectory gives a 1-D vertex estimate, and
#   - the local GP mean is maximised along the extended trajectory.
# Both are 1-D diagnostics, not proof of a 2-D optimum.

trajectory_data = df[df["Round"].isin(TRAJECTORY_ROUNDS)].sort_values("Round")
X_trajectory = trajectory_data[feature_cols].to_numpy()
y_trajectory = trajectory_data["y"].to_numpy()
trajectory_anchor = incumbent_x.copy()

# Direction: signed first principal component, scaled to the median spacing
# between successive on-trajectory observations (so t is in "steps").
trajectory_center = X_trajectory.mean(axis=0)
trajectory_direction = np.linalg.svd(X_trajectory - trajectory_center, full_matrices=False)[2][0]
if np.dot(trajectory_direction, X_trajectory[-1] - X_trajectory[0]) < 0:
    trajectory_direction = -trajectory_direction
projected = (X_trajectory - trajectory_center) @ trajectory_direction
TRAJECTORY_STEP = trajectory_direction * float(np.median(np.diff(projected)))
print(f"Observed rounds: {trajectory_data['Round'].tolist()}")
print(f"Data-fitted trajectory step: {TRAJECTORY_STEP}")

# Quadratic y(t) = a t^2 + b t + c (a < 0 needed for an interior maximum)
t_obs = ((X_trajectory - trajectory_anchor) @ TRAJECTORY_STEP) / (TRAJECTORY_STEP @ TRAJECTORY_STEP)
a, b, c = np.polyfit(t_obs, y_trajectory, 2)
t_vertex = -b / (2.0 * a) if a < 0 else np.nan
x_vertex = trajectory_anchor + t_vertex * TRAJECTORY_STEP if np.isfinite(t_vertex) else np.full(2, np.nan)
y_vertex = np.polyval([a, b, c], t_vertex) if np.isfinite(t_vertex) else np.nan
print(f"Quadratic: y(t) = {a:.6e} t² + {b:.6e} t + {c:.6e}")
print(f"Quadratic vertex: t = {t_vertex:.6f}, x = ({x_vertex[0]:.6f}, {x_vertex[1]:.6f}), y = {y_vertex:.6e}")

# Local GP mean along the extended trajectory (anchored at the incumbent)
t_extended = np.linspace(EXTENDED_T_MIN, EXTENDED_T_MAX, EXTENDED_GRID_POINTS)
X_extended = trajectory_anchor + t_extended[:, None] * TRAJECTORY_STEP
gp_mean_ext, gp_std_ext = gp_local.predict(X_extended, return_std=True)
gp_best_idx = int(np.argmax(gp_mean_ext))
gp_best_t, gp_best_x = t_extended[gp_best_idx], X_extended[gp_best_idx]
gp_at_boundary = gp_best_idx in (0, len(t_extended) - 1)
print(f"Extended GP mean maximum (t in [{EXTENDED_T_MIN}, {EXTENDED_T_MAX}]): t = {gp_best_t:.6f}, "
      f"x = ({gp_best_x[0]:.6f}, {gp_best_x[1]:.6f}), transformed mean = {gp_mean_ext[gp_best_idx]:.6f} "
      f"± {gp_std_ext[gp_best_idx]:.6f}, on boundary: {gp_at_boundary}")

trajectory_comparison = pd.DataFrame([
    {"Candidate": "Current incumbent", "t": 0.0, "x1": trajectory_anchor[0], "x2": trajectory_anchor[1]},
    {"Candidate": "Full next step", "t": 1.0, "x1": trajectory_anchor[0] + TRAJECTORY_STEP[0], "x2": trajectory_anchor[1] + TRAJECTORY_STEP[1]},
    {"Candidate": "Quadratic vertex", "t": t_vertex, "x1": x_vertex[0], "x2": x_vertex[1]},
    {"Candidate": "Extended GP mean maximum", "t": gp_best_t, "x1": gp_best_x[0], "x2": gp_best_x[1]},
])
print("\nTRAJECTORY CANDIDATES")
print(trajectory_comparison.to_string(index=False, float_format=lambda v: f"{v:.6f}"))

Observed rounds: [6, 7, 9, 10, 11]
Data-fitted trajectory step: [0.0004004 0.0001992]
Quadratic: y(t) = -2.967409e-08 t² + 1.522604e-08 t + 3.543069e-06
Quadratic vertex: t = 0.256554, x = (0.671653, 0.700821), y = 3.545022e-06
Extended GP mean maximum (t in [0.0, 5.0]): t = 0.095000, x = (0.671588, 0.700789), transformed mean = 29.124667 ± 0.002090, on boundary: False

TRAJECTORY CANDIDATES
               Candidate        t       x1       x2
       Current incumbent 0.000000 0.671550 0.700770
          Full next step 1.000000 0.671950 0.700969
        Quadratic vertex 0.256554 0.671653 0.700821
Extended GP mean maximum 0.095000 0.671588 0.700789


In [9]:
# =============================================================================
# TWO-DIMENSIONAL LOCAL EXPLORATION AND RECOMMENDED QUERY
# =============================================================================
# The successful observations are concentrated along one narrow direction, so
# the location of the maximum in the surrounding 2-D space is uncertain. This
# table compares GP predictions for small moves from the incumbent along and
# perpendicular to the trajectory, and along each input. Off-axis predictions
# are less reliable because off-axis high-value observations are scarce; the
# table does not replace an experimental evaluation.

perpendicular = np.array([-trajectory_direction[1], trajectory_direction[0]])
directions = [("Along +", trajectory_direction), ("Along -", -trajectory_direction),
              ("Perpendicular +", perpendicular), ("Perpendicular -", -perpendicular),
              ("x1 +", np.array([1.0, 0.0])), ("x1 -", np.array([-1.0, 0.0])),
              ("x2 +", np.array([0.0, 1.0])), ("x2 -", np.array([0.0, -1.0]))]

rows = []
for radius in EXPLORATION_RADII:
    for label, direction in directions:
        point = np.clip(incumbent_x + radius * direction, 0.0, 1.0)
        mean, std = gp_local.predict(point[None, :], return_std=True)
        rows.append({"Radius": radius, "Direction": label, "x1": point[0], "x2": point[1],
                     "GP_Transformed_Mean": mean[0], "GP_Std": std[0],
                     "Distance_to_Nearest_Observation": np.linalg.norm(X - point, axis=1).min()})
exploration_df = pd.DataFrame(rows)

print("LOCAL 2-D EXPLORATION: MODEL-BASED DIRECTIONAL COMPARISON")
print(exploration_df.to_string(index=False, float_format=lambda v: f"{v:.6f}"))

# Recommended query: the configured direction and radius
chosen = exploration_df[(exploration_df["Direction"] == RECOMMENDED_DIRECTION)
                        & np.isclose(exploration_df["Radius"], RECOMMENDED_RADIUS)].iloc[0]
x_next1 = chosen[feature_cols].to_numpy(dtype=float)
print(f"\nRecommended next query ({RECOMMENDED_DIRECTION}, radius {RECOMMENDED_RADIUS}):")
print(format_point(x_next1))

LOCAL 2-D EXPLORATION: MODEL-BASED DIRECTIONAL COMPARISON
  Radius       Direction       x1       x2  GP_Transformed_Mean   GP_Std  Distance_to_Nearest_Observation
0.000500         Along + 0.671998 0.700993            29.115105 0.005463                         0.000500
0.000500         Along - 0.671102 0.700547            29.110804 0.002256                         0.000111
0.000500 Perpendicular + 0.671327 0.701218            28.694364 0.319677                         0.000500
0.000500 Perpendicular - 0.671773 0.700322            29.456156 0.323872                         0.000500
0.000500            x1 + 0.672050 0.700770            29.296564 0.141399                         0.000500
0.000500            x1 - 0.671050 0.700770            28.952577 0.141110                         0.000227
0.000500            x2 + 0.671550 0.701270            28.724163 0.287682                         0.000500
0.000500            x2 - 0.671550 0.700270            29.403145 0.290675                      